# Tarea 2 — Vbak_Autoloader_Ingesta

Auto Loader incremental: procesa SOLO los archivos nuevos desde el último checkpoint.
Depende de: `Vbak_Generar_Batch`

In [0]:
from pyspark.sql.functions import current_timestamp, col

CATALOG = "laboratory_sap_dev"
SCHEMA  = "sap_course"
SAP_DATA_PATH   = f"/Volumes/{CATALOG}/bronze/curso_databricks"
LANDING_PATH    = f"{SAP_DATA_PATH}/landing/vbak"
CHECKPOINT_PATH = f"{SAP_DATA_PATH}/_checkpoints/vbak_al"
TARGET_TABLE    = f"{CATALOG}.{SCHEMA}.vbak_bronze_autoloader"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("pathGlobFilter", "vbak_*.csv")
    .load(LANDING_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_autoloader_ts", current_timestamp()))

query = (stream.writeStream
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE))
query.awaitTermination()

n = spark.table(TARGET_TABLE).count()
print(f"OK: Auto Loader -> {TARGET_TABLE}: {n:,} registros totales")